![OneHealth DataSpace](https://bigdata.dataspace.cesga.es/static/images/public/imagotipo.png)

> **⚠️ ADVERTENCIA**: actualmente la plataforma está en fase de pueba. Agradecemos que nos hagáis llegar cualquier comentario a onehealth@cesga.es. 

# Profundidad de la captura da raya

En este notebook vamos ver como analizar a profundidad de la captura de la raya a partir de los datos de caputras de DIVERSIMAR.

Para comezar, importaremos algunos módulos que en general nos serán de utilidad:

In [ ]:
import onehealth
import pyspark.sql.functions as F
from pyspark.sql.functions import col, expr
import pandas as pd
import numpy as np

## Carga de datos

Ahora cargaremos el conjunto de datos de `capturas`:

In [ ]:
capturas = onehealth.load('especies/ieo_capturas')


## Exploración

Vemos cuantas mediciones tenemos:

In [5]:
capturas.count()

337746

Podemos ver las propiedades disponibles para cada medición:

In [ ]:
onehealth.describe('especies/ieo_capturas')

Valor,Unidad,Tipo,Descripción
id_observation,,string,Identificación de la observación.
aphia_id_code,,string,Código de identificación taxonómica (Aphia)
cod_ieem,,string,Código IEEM de la observación.
id_spezie,,string,Identificación de la especie.
x3a_code,,string,Nombre científico de la especie.
scientific_name,,string,Nombre FAO en español.
spanish_fao_name,,string,Nombre FAO en inglés.
english_fao_name,,string,Proyecto relacionado con la observación.
project,,string,Fecha de la observación.
obs_date,,string,Número de individuos observados.


Puede ser de ayuda explorar las primeras mediciones:

In [8]:
capturas.limit(5).toPandas()

,id_observation,aphia_id_code,cod_ieem,id_spezie,x3a_code,scientific_name,spanish_fao_name,english_fao_name,project,obs_date,individuals,weight_gr,weight_kg,depth,weight_gr_dum,camp,long_4326,lat_4326,geom
0,22,140600,6483,125,EOI,Eledone cirrhosa,Pulpo blanco,Horned octopus,"spezies, mapdescar",2020-09-29,9,1070,1.07,230,1070,N19,-9.5382,42.784,0101000020E6100000166A4DF38E1323C03108AC1C5A64...
1,103,140600,6483,125,EOI,Eledone cirrhosa,Pulpo blanco,Horned octopus,"spezies, mapdescar",2020-09-29,4,230,0.23,114,230,N10,-9.0427,41.974,0101000020E610000012143FC6DC1522C0E9263108ACFC...
2,104,140600,6483,125,EOI,Eledone cirrhosa,Pulpo blanco,Horned octopus,"spezies, mapdescar",2020-09-29,18,1448,1.448,138,1448,N10,-9.1271,42.0547,0101000020E6100000A9A44E40134122C0AC8BDB680007...
3,105,140600,6483,125,EOI,Eledone cirrhosa,Pulpo blanco,Horned octopus,"spezies, mapdescar",2020-09-29,10,475,0.475,104,475,N10,-8.9856,42.0859,0101000020E61000000DE02D90A0F821C0FB5C6DC5FE0A...
4,106,140600,6483,125,EOI,Eledone cirrhosa,Pulpo blanco,Horned octopus,"spezies, mapdescar",2020-09-29,7,1180,1.18,133,1180,N10,-9.0694,42.1569,0101000020E61000004FAF9465882322C0075F984C1514...


## Filtrado de los datos

Vamos filtrar los datos para quedarnos solo con los datos de captura para la *Raya común* (`scientific_name`: *Raja clavata*, `spanish_fao_name`: *Raya de clavos*, `cod_ieem` 4796) es donde hay información sobre el número de individuos capturados:

In [ ]:
raya = capturas.filter('scientific_name = "Raja clavata" AND individuals != 0')

Veamos cuantas mediciones tenemos tras filtrarlo:

In [ ]:
raya.count()

464

Podemos explorar las primeiras mediciones:

In [ ]:
raya.limit(5).toPandas()

,id_observation,aphia_id_code,cod_ieem,id_spezie,x3a_code,scientific_name,spanish_fao_name,english_fao_name,project,obs_date,individuals,weight_gr,weight_kg,depth,weight_gr_dum,camp,long_4326,lat_4326,geom
0,258063,105883,4796,322,RJC,Raja clavata,Raya de clavos,Thornback ray,"espezies, mapdescar",NULL,2,70,0.07,114,NULL,N10,-9.0427,41.974,0101000020E610000012143FC6DC1522C0E9263108ACFC...
1,258064,105883,4796,322,RJC,Raja clavata,Raya de clavos,Thornback ray,"espezies, mapdescar",NULL,1,12.409,0.012,138,NULL,N10,-9.1271,42.0547,0101000020E6100000A9A44E40134122C0AC8BDB680007...
2,258065,105883,4796,322,RJC,Raja clavata,Raya de clavos,Thornback ray,"espezies, mapdescar",NULL,1,4350,4.35,104,NULL,N10,-8.9856,42.0859,0101000020E61000000DE02D90A0F821C0FB5C6DC5FE0A...
3,258069,105883,4796,322,RJC,Raja clavata,Raya de clavos,Thornback ray,"espezies, mapdescar",NULL,1,142,0.142,192,NULL,N10,-9.2827,42.1152,0101000020E61000008D28ED0DBE9022C0E63FA4DFBE0E...
4,258070,105883,4796,322,RJC,Raja clavata,Raya de clavos,Thornback ray,"espezies, mapdescar",NULL,6,15400,15.4,99,NULL,N10,-8.9702,42.2326,0101000020E61000008D28ED0DBEF021C0598638D6C51D...


## Visualización

Como ya realizamos el filtrado de datos el resultado es de un tamaño manejable sin necesidad de procesamiento en paralelo, lo que haremos es quedarnos solo con los campos que nos interesan y convertir el resultado a un dataframe de pandas:

In [ ]:
df = raya.select('depth', 'individuals', 'weight_gr').toPandas()

A partir de ahora ya podemos trabajar de la misma manera que haríamos en local.

Primeiro convertiremos los datos de profundidad e individuos que están en texto a formato numérico:

In [ ]:
df['profundidad'] = pd.to_numeric(df['depth'])
df['individuos'] = pd.to_numeric(df['individuals'])

Calculemos el histograma:

In [ ]:
edges = list(range(0, 1550, 50))
df['bins'] = pd.cut(df['profundidad'], bins=edges, right=False)
histogram = df.groupby('bins')['individuos'].sum().reset_index()
histogram['bin_left'] = histogram['bins'].apply(lambda x: x.left)

/tmp/ipykernel_18928/760140801.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  histogram = df.groupby('bins')['individuos'].sum().reset_index()


Ahora podemos generar una visualización usando `bokeh` que nos permitirá interactuar de la misma manera que el navegador. Importemos las librarías necesarias:

In [18]:
from bokeh.palettes import Viridis256
from bokeh.transform import linear_cmap
from bokeh.plotting import figure, curdoc
from bokeh.models import ColumnDataSource, ColorBar, Span, Band, HoverTool, Button, Select, Legend
from bokeh.plotting import figure, output_file, show
from bokeh.io import output_notebook, push_notebook

In [ ]:
cds = ColumnDataSource(data=dict(top=histogram['individuos'], left=edges[:-1], right=edges[1:]))

p = figure(title=f'Profundidad de captura da Raya', 
           x_axis_label='Profundidad (m)', y_axis_label='Capturas',
           height=800, width=900)
color_mapper = linear_cmap(field_name='top', palette=Viridis256, low=min(histogram['individuos']), high=max(histogram['individuos']))
bars = p.vbar(x='left', top='top', bottom=0, width=50, source=cds, line_color='white',
       fill_color=color_mapper)
hover = HoverTool(tooltips=[("capturas", "@top")], renderers=[bars])
p.add_tools(hover)

output_notebook()
show(p)

Loading BokehJS ...